## 1. Environment & Path Setup
This cell dynamically checks if you are running in the cloud natively (Google Colab) or locally (Antigravity IDE), and sets the root working directory accordingly without crashing.

In [ ]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    # Google Drive structure assumption
    PROJECT_PATH = '/content/drive/MyDrive/androidrec/Android_Recommender_System'
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"\nSuccessfully targeted project directory: {os.getcwd()}")
    else:
        print(f"\n[ERROR] Could not find {PROJECT_PATH} in Google Drive.")
        print("Please upload the Android_Recommender_System folder directly into your root MyDrive.")
else:
    print("Running locally. Retaining current project root:")
    print(os.getcwd())

Running in Google Colab. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Successfully targeted project directory: /content/drive/MyDrive/androidrec/Android_Recommender_System


## 2. Dependency Installation & GPU Verification
Google Colab already provisions `torch`, `numpy`, and `scikit-learn`. Forcing manual re-installs of Torch pip wheels can destroy cloud GPU bindings. This cell only installs what is strictly missing (`torch-geometric`) and performs safety checks.

In [10]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("\n[CRITICAL WARNING] No GPU detected! Your graph neural network will train extremely slowly.")
    print("If on Google Colab: Go to 'Runtime' > 'Change runtime type' > Select 'T4 GPU'\n")

if IN_COLAB:
    # Quiet installation of remaining libraries required by the Colab cloud
    !pip install -q torch-geometric scikit-surprise joblib


PyTorch Version: 2.10.0+cu128
CUDA Available: True
GPU Name: Tesla T4


## 3. Data Pipeline Intercept Check
The graph requires `.npz` binary matrices to execute. If they haven't been constructed yet (e.g. fresh VM instance), this safely auto-triggers the build script.

In [11]:
!python variant_graph/graph_build_npy.py


Loading raw myket.csv...
/content/drive/MyDrive/androidrec/Android_Recommender_System/variant_graph/graph_build_npy.py:27: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df_raw = pd.read_csv("data/myket.csv", index_col=False, on_bad_lines="skip")
  Raw rows:  694,121
  Raw users: 10,000
  Raw apps:  7,988

After encoding:
Interactions: 694,121
Users: 10,000  (max=9999)
Apps:  7,988  (max=7987)
Saved: user2idx.pkl, app2idx.pkl, idx2app.pkl

Building refined app features from CSV ...
  Total apps processed: 7,988
  Final feature dimension: 566
  Apps using zero fallback (missing metadata): 382
  Saved: data/app_info_sample.npy

N = 5 held-out interactions per user per split
  Train: 594,121 interactions | 10,000 users
  Val:   50,000 interactions  | 10,000 users
  Test:  50,000 interactions  | 10,000 users

Sample user 0 train history (35 interactions):
  [799, 6242, 2728, 7158, 4214, 3688, 5418, 5948, 7104, 51

## 4. Execute LightGCN on GPU
Fires the CUDA-optimized graph variant script directly locally. Metrics will output mathematically identically.

In [12]:
!python variant_graph/variant_lightgcn_colab.py

Using device: cuda
GPU Name: Tesla T4

Loading pipeline outputs...
Users: 10000 | Items: 7988 | Total Nodes: 17988

Building the bipartite graph...
Total graph edges after adding reverse links: 948,310

Starting LightGCN training for 500 epochs on cuda...
Epoch 001/500 | BPR Loss: 0.6931
Epoch 025/500 | BPR Loss: 0.6502
Epoch 050/500 | BPR Loss: 0.5048
Epoch 075/500 | BPR Loss: 0.4104
Epoch 100/500 | BPR Loss: 0.3655
Epoch 125/500 | BPR Loss: 0.3335
Epoch 150/500 | BPR Loss: 0.3079
Epoch 175/500 | BPR Loss: 0.2868
Epoch 200/500 | BPR Loss: 0.2685
Epoch 225/500 | BPR Loss: 0.2533
Epoch 250/500 | BPR Loss: 0.2389
Epoch 275/500 | BPR Loss: 0.2266
Epoch 300/500 | BPR Loss: 0.2143
Epoch 325/500 | BPR Loss: 0.2029
Epoch 350/500 | BPR Loss: 0.1935
Epoch 375/500 | BPR Loss: 0.1839
Epoch 400/500 | BPR Loss: 0.1762
Epoch 425/500 | BPR Loss: 0.1686
Epoch 450/500 | BPR Loss: 0.1625
Epoch 475/500 | BPR Loss: 0.1585
Epoch 500/500 | BPR Loss: 0.1534
Training completed in 146.3 seconds

Generating Top

In [16]:
!python variant_graph/variant_lightgcn_metadata_colab.py

Using device: cuda
GPU Name: Tesla T4

Loading pipeline outputs...
Users: 10000 | Items: 7988 | Total Nodes: 17988

Building the bipartite graph...
Total graph edges after adding reverse links: 948,310

Starting LightGCN training for 500 epochs on cuda...
Epoch 001/500 | BPR Loss: 0.6929
Epoch 025/500 | BPR Loss: 0.4288
Epoch 050/500 | BPR Loss: 0.2896
Epoch 075/500 | BPR Loss: 0.2403
Epoch 100/500 | BPR Loss: 0.2038
Epoch 125/500 | BPR Loss: 0.1755
Epoch 150/500 | BPR Loss: 0.1569
Epoch 175/500 | BPR Loss: 0.1442
Epoch 200/500 | BPR Loss: 0.1353
Epoch 225/500 | BPR Loss: 0.1449
Epoch 250/500 | BPR Loss: 0.1312
Epoch 275/500 | BPR Loss: 0.1242
Epoch 300/500 | BPR Loss: 0.1168
Epoch 325/500 | BPR Loss: 0.1115
Epoch 350/500 | BPR Loss: 0.1074
Epoch 375/500 | BPR Loss: 0.1046
Epoch 400/500 | BPR Loss: 0.0999
Epoch 425/500 | BPR Loss: 0.0959
Epoch 450/500 | BPR Loss: 0.0931
Epoch 475/500 | BPR Loss: 0.0885
Epoch 500/500 | BPR Loss: 0.0856
Training completed in 151.2 seconds

Generating Top